In [7]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import math

In [11]:
import torch
import torch.nn as nn
import math

t = torch.tensor([[0.1,0.2,0.3,0.4],
                  [0.5,0.6,0.7,0.8],
                  [0.9,1.0,1.1,1.2]], dtype=torch.float32)

class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super(SelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key   = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.embed_dim)
        weights = self.softmax(scores)
        output = torch.matmul(weights, V)
        return output

embed_dim = t.shape[1]  
model = SelfAttention(embed_dim)
output = model(t)
print(output)


tensor([[ 0.0778, -0.0637,  0.8834,  1.0643],
        [ 0.0779, -0.0640,  0.8845,  1.0652],
        [ 0.0779, -0.0643,  0.8855,  1.0661]], grad_fn=<MmBackward0>)


In [13]:


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = F.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        batch_size, num_heads, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output
d_model = 512
num_heads = 8
batch_size = 32
seq_length = 10

mha = MultiHeadAttention(d_model, num_heads)

query = torch.randn(batch_size, seq_length, d_model)
key = torch.randn(batch_size, seq_length, d_model)
value = torch.randn(batch_size, seq_length, d_model)

mask = torch.ones(batch_size, 1, 1, seq_length)

output = mha(query, key, value, mask=mask)

print(output)
print(f"Input shape: {query.shape}")
print(f"Output shape: {output.shape}")


tensor([[[-0.0471,  0.0175,  0.0929,  ...,  0.2126, -0.0304,  0.0105],
         [-0.0343, -0.0073,  0.0991,  ...,  0.2479, -0.1083,  0.0587],
         [ 0.1033,  0.0135,  0.0939,  ...,  0.2079, -0.0818,  0.1048],
         ...,
         [-0.0146,  0.0189,  0.1057,  ...,  0.2004, -0.1006,  0.0398],
         [-0.0641, -0.0188,  0.1353,  ...,  0.2367, -0.0993,  0.0728],
         [ 0.0230,  0.0720,  0.1253,  ...,  0.1946, -0.0668,  0.0849]],

        [[-0.2607, -0.0100,  0.0664,  ..., -0.1335, -0.1225,  0.2350],
         [-0.2513, -0.0006,  0.1189,  ..., -0.1037, -0.0639,  0.2170],
         [-0.2269, -0.0271,  0.0863,  ..., -0.1264, -0.0857,  0.2459],
         ...,
         [-0.1660, -0.0130,  0.0583,  ..., -0.1460, -0.1075,  0.2579],
         [-0.1993,  0.0425,  0.1005,  ..., -0.1437, -0.0927,  0.2865],
         [-0.2012, -0.1135,  0.0339,  ..., -0.1130, -0.0697,  0.2086]],

        [[-0.1292, -0.1545,  0.0450,  ...,  0.0204, -0.1474, -0.1448],
         [-0.0802, -0.1496,  0.0055,  ..., -0